In [ ]:
# CÉLULA DE TESTE DA BIBLIOTECA CYVCF2

from cyvcf2 import VCF

FILE_NAME = 'Y.vcf'
FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

# Abre o arquivo .vcf (ou .vcf.gz)
vcf = VCF(FILE_PATH)

#Caso se queira visualizar somente o header
#print(vcf.raw_header)

#visualização das amostras participantes
print("amostras: ", vcf.samples)

#Iterando sobre as variantes(linhas do Vcf)
for variant in vcf:

  #Impressão da linha completa
  print("Variante: ", variant)
  
  #Leitura de dados básicos
  cromossomo = variant.CHROM
  posicao = variant.POS
  id = variant.ID
  ref = variant.REF
  alt = variant.ALT
  quality = variant.QUAL

  #Acessando o campo INFO
  dp = variant.INFO.get('DP')
  ac = variant.INFO.get('AC')
  af = variant.INFO.get('AF')

  #Acessando os genótipos das amostras
  genotipos = variant.genotypes
  gt_types = variant.gt_types
  gt_ref_depths = variant.gt_ref_depths
  gt_alt_depths = variant.gt_alt_depths
  gt_phases = variant.gt_phases
  gt_quals = variant.gt_quals
  gt_bases = variant.gt_bases

  print(f'CHROM: {cromossomo}\nPOS: {posicao}\nID: {id}\nREF: {ref}\nALT: {alt}\nQUAL: {quality:.2f}\n')
  print(genotipos)
  # print(gt_types)
  # print(gt_alt_depths)
  # print(gt_ref_depths)
  # print(gt_phases)
  # print(gt_quals)
  # print(gt_bases)

  #Forma de acessar informações especificas do campo FORMAT (para uma determinada amostra deve-se especificar o idx) 
  # (ex: os resultados de DP daquela amostra em todas as variantes)
  sample_idx = 1
  dp_array = variant.format('DP')
  dp = dp_array[sample_idx].tolist() if dp_array is not None else None

  #Para imprimir apenas o primeiro
  break

vcf.close()


In [ ]:
import pandas as pd
import numpy as np
import os
from enum import Enum
from cyvcf2 import VCF


class BASIC_COLS(Enum):
    CHROM = 0
    POS = 1
    ID = 2
    REF = 3
    ALT = 4
    QUAL = 5
    FILTER = 6
    INFO = 7  
    FORMAT = 8


def vcf_to_df_raw(vcf_path):
    vcf_file = VCF(vcf_path)

    cmd = "zgrep '^#' " + vcf_path + "|tail -n 1"
    cols = os.popen(cmd).read().strip('#').strip('\n').split('\t')
    
    data = []
    
    for variant in vcf_file:
        raw_line = str(variant).rstrip('\n').split('\t')
        data.append(raw_line)

    df = pd.DataFrame(data, columns=cols)

    vcf_file.close()
    return df

def vcf_to_df_filtered_INFO(vcf_path):
    vcf_file = VCF(vcf_path)
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
        ('FORMAT', '')
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        cols_tuples.append((sample, ''))

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
            raw_line[BASIC_COLS.FORMAT.value] 
        ]

        fltrd_line.extend(raw_line[BASIC_COLS.FORMAT.value+1:])
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    vcf_file.close()
    return df

def vcf_to_df_filtered_Samples(vcf_path):
    vcf_file = VCF(vcf_path)
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('QUAL', ''),
        ('FILTER', ''),
        ('INFO', 'DP'),  
        ('INFO', 'GT'),  
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        sample_tuple = [
            (sample, 'GT'),
            (sample, 'AF'),
            (sample, 'DP'),
        ]
        cols_tuples.extend(sample_tuple)

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        fltrd_line = [
            variant.CHROM,
            variant.POS,                                   
            variant.REF,
            ",".join(variant.ALT) if variant.ALT else ".",  # ALT é uma lista, transformamos em string separada por vírgula
            variant.QUAL,                                   
            variant.FILTER if variant.FILTER else "PASS",   # cyvcf2 retorna None se for PASS
            variant.INFO.get('DP'),                        
            variant.INFO.get('GT'),    
        ]

        # Extração com cyvcf2. Retorna arrays ou None.
        genotypes = variant.genotypes # array com os GT's
        af_array = variant.format('AF')
        dp_array = variant.format('DP')

        samples_fltrd_line = []
        for i in range(len(vcf_file.samples)):
            # Trocamos o "." por None. Isso permite que o Pandas use NaN e mantenha a coluna numérica!
            samples_fltrd_line.extend([
                genotypes[i] if genotypes is not None else None,
                af_array[i][0] if af_array is not None else None,
                dp_array[i][0] if dp_array is not None else None
            ])
        
        # Une as amostras ao restante das informações
        fltrd_line.extend(samples_fltrd_line)
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    vcf_file.close()
    return df

def df_to_csv(df, file_name):
    output_folder = 'output'
    name = file_name.replace('.vcf', '').replace('.gz', '') + '.csv'
    folder_path = os.path.join(output_folder, name)
    os.makedirs(output_folder, exist_ok=True)
    df.to_csv(folder_path, index=False)


In [2]:
# FILE_NAME = 'Y.vcf'
FILE_NAME = 'Cyberseg_chr21.vcf'
FILE_PATH = f'/home/marcela/IC/files/{FILE_NAME}'

df_raw = vcf_to_df_raw(FILE_PATH)
df_filtered_INFO = vcf_to_df_filtered_INFO(FILE_PATH)
df_filtered_Samples = vcf_to_df_filtered_Samples(FILE_PATH)

print(df_raw.shape)
print(df_filtered_INFO.shape)
print(df_filtered_Samples.shape)

[W::vcf_parse_format_dict2] FORMAT 'MB' at 21:17441798 is not defined in the header, assuming Type=String
[W::vcf_parse_filter] FILTER 'MosaicLowAF' is not defined in the header
[W::vcf_parse_format_dict2] FORMAT 'MB' at 21:17441798 is not defined in the header, assuming Type=String
[W::vcf_parse_filter] FILTER 'MosaicLowAF' is not defined in the header
[W::vcf_parse_format_dict2] FORMAT 'MB' at 21:17441798 is not defined in the header, assuming Type=String
[W::vcf_parse_filter] FILTER 'MosaicLowAF' is not defined in the header


(7491, 406)
(7491, 405)
(7491, 1199)


In [ ]:
# Célula para salvar no formato original de rows x cols (OBS: pela qtd de cols pode não abrir para visualização!)
import sqlite3

def save_in_db(df, file_name):

    table_name = file_name.replace('.vcf', '') 

    # Achatamento para o caso de colunas multiindex
    df.columns = [
        '_'.join(col).strip('_') if isinstance(col, tuple) else col 
        for col in df.columns.values
    ]

    # Conectar e salvar no banco
    with sqlite3.connect('genomic.db') as connection:
        print(f"Successfully connected to database!")
        
        df.to_sql(table_name, connection, if_exists='replace', index=False)
        
        print(f"Data from {file_name} successfully saved!")

In [3]:
# Célula para tabelas que possuem MultiIndex nas amostras

import sqlite3

def save_split_table_db(df_input, base_name, multi_index=False):
    # Tabela de variantes
    df_variantes = df_input.iloc[:, :8].copy()

    if(multi_index):
        df_variantes.columns = [
            f"{col[0]}_{col[1]}" if col[1] else col[0] 
            for col in df_variantes.columns
        ]

    df_variantes.insert(0, 'ID_VARIANTE', range(1, len(df_variantes) + 1))

    # Tabela de amostras
    df_amostras = df_input.iloc[:, 8:].copy()

    df_amostras = df_amostras.stack(level=0, future_stack=True).reset_index()

    df_amostras = df_amostras.rename(columns={'level_0': 'ID_VARIANTE', 'level_1': 'AMOSTRA'})
    df_amostras['ID_VARIANTE'] = df_amostras['ID_VARIANTE'] + 1

    # if 'GT' in df_amostras.columns:
    #     df_amostras['GT'] = df_amostras['GT'].astype(str)
    if 'GT' in df_amostras.columns:
        # Transforma a lista em texto, mas se o dado for vazio (None), mantém vazio.
        df_amostras['GT'] = df_amostras['GT'].apply(lambda x: str(x) if x is not None else None)

    # Conexão com o banco de dados
    with sqlite3.connect('genomic.db') as connection:
        print("Conectado ao banco de dados!")
        
        df_variantes.to_sql(f"{base_name}_Variantes", connection, if_exists='replace', index=False)
        df_amostras.to_sql(f"{base_name}_Amostras", connection, if_exists='replace', index=False)
        
        print(f"Sucesso! As tabelas '{base_name}_Variantes' e '{base_name}_Amostras' foram criadas.")

In [ ]:
df_raw = vcf_to_df_raw(FILE_PATH)
base_name = FILE_NAME.replace('.vcf', '').replace('.gz', '')
save_split_table_db(df_raw, f"Raw_{base_name}")

In [4]:
df_multi = vcf_to_df_filtered_Samples(FILE_PATH)
base_name = FILE_NAME.replace('.vcf', '').replace('.gz', '')
save_split_table_db(df_multi, f"Fltrd_{base_name}", True)

[W::vcf_parse_format_dict2] FORMAT 'MB' at 21:17441798 is not defined in the header, assuming Type=String
[W::vcf_parse_filter] FILTER 'MosaicLowAF' is not defined in the header


Conectado ao banco de dados!
Sucesso! As tabelas 'Fltrd_Cyberseg_chr21_Variantes' e 'Fltrd_Cyberseg_chr21_Amostras' foram criadas.
